# Feature importance — NO2 exceedance model

Which features does the model actually rely on? Computed via **permutation importance** on the held-out test set (not the default tree `feature_importances_`, which can be misleading with correlated features like temperature/month).

This reuses the exact same data loading, feature engineering, and temporal split as `train_no2_exceedance.py` — see that file for the full reasoning behind each choice (target threshold, walk-forward validation, etc.).

In [ ]:
import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

from train_no2_exceedance import (
    THRESHOLD_UG_M3, HORIZON_HOURS, TEST_DAYS,
    OPENAQ_PATH, WEATHER_PATH,
    load_no2, load_weather, build_features,
)

## Rebuild the exact same dataset and temporal split

Copied from `train_no2_exceedance.main()` rather than re-importing it, since that
function does everything (load, train, evaluate, save) in one block. This is
just the data-prep portion, stopped right before modelling.

In [ ]:
no2 = load_no2()
weather = load_weather()
df = no2.merge(weather, on=["location_id", "datetime"], how="left")
df = build_features(df)
df = df.dropna(subset=["target"])
df["target"] = df["target"].astype(int)
df["location_id"] = df["location_id"].astype("category")
df = df.sort_values("datetime").reset_index(drop=True)

cutoff = df["datetime"].max() - pd.Timedelta(days=TEST_DAYS)
train = df[df["datetime"] < cutoff]
test = df[df["datetime"] >= cutoff]

numeric_cols = [
    "hour", "dayofweek", "month", "is_weekend", "is_holiday",
    "no2_lag1h", "no2_lag3h", "no2_lag6h", "no2_lag24h", "no2_roll24h_mean",
    "temperature_2m", "relative_humidity_2m", "wind_speed_10m",
    "wind_dir_sin", "wind_dir_cos", "precipitation", "surface_pressure", "cloud_cover",
]
feature_cols = numeric_cols + ["location_id"]

X_train, y_train = train[feature_cols].reset_index(drop=True), train["target"].reset_index(drop=True)
X_test, y_test = test[feature_cols], test["target"]

print(f"train: {len(X_train):,} rows, test: {len(X_test):,} rows")

In [ ]:
# Same model as the main training script, fit on train only (matches the
# honest evaluation split -- NOT the final production model, which is
# refit on everything).
hgb = HistGradientBoostingClassifier(random_state=42, categorical_features=["location_id"])
hgb.fit(X_train, y_train)

## Your turn: permutation importance

Compute permutation importance of `hgb` on `(X_test, y_test)`, scoring on `roc_auc`,
with enough repeats to be stable (not a one-shot shuffle).

Then sort features by mean importance (descending) and print them next to their
name from `feature_cols`.

In [ ]:
# TODO: your code here


## Visualize

A horizontal bar chart, sorted, is the standard way to present this --
use `result.importances_mean` and `result.importances_std` (as an error bar)
once the cell above gives you `result`.

In [ ]:
# TODO: your code here
